<a href="https://colab.research.google.com/github/cpython-projects/python_da_17_03_26/blob/main/lesson_24_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
DB_USER = "prog_academy_da_157g_user"
DB_PASSWORD = "XTE8IHOEtQZ5GasTkaOjcGMKYTha6LeO"
DB_HOST = "dpg-d8t957gg4nts73da5bg0-a.frankfurt-postgres.render.com"
DB_PORT = "5432"
DB_NAME = "prog_academy_da_157g"

In [2]:
url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

In [4]:
from sqlalchemy import create_engine

engine = create_engine(url)

import pandas as pd

1. **Загальна кількість установок**

In [7]:
query = """
SELECT COUNT(*) AS total_installs
FROM app_sessions;
"""
df = pd.read_sql(query, engine)
df

,total_installs
0,500


2. **На скількох унікальних пристроях встановили застосунок**

In [11]:
query = """
SELECT COUNT(DISTINCT device_code) AS unique_installs
FROM app_sessions;
"""

df = pd.read_sql(query, engine)
df

,unique_installs
0,500


3. **Скільки пристроїв взаємодіяли з каталогом товарів**

In [12]:
query = """
SELECT COUNT(DISTINCT device_code) AS number_of_devices
FROM product_views_log;
"""

df = pd.read_sql(query, engine)
df

,number_of_devices
0,500


4. **Скільки авторизованих користувачів**

In [13]:
query = """
SELECT COUNT(user_uuid) as auth_users
FROM devices_users_map;
"""

df = pd.read_sql(query, engine)
df

,auth_users
0,300


5. **Скільки користувачів здійснили хоча б одну покупку**

In [14]:
query = """
SELECT COUNT(DISTINCT user_uuid) as number_of_customers
FROM orders_log;
"""
df = pd.read_sql(query, engine)
df

,number_of_customers
0,300


6. **Скільки пристроїв були авторизовані (пов’язані з user\_uuid)**

In [15]:
query = """
SELECT COUNT(device_code) as auth_devices
FROM devices_users_map;
"""

df = pd.read_sql(query, engine)
df

,auth_devices
0,300


7. **Конверсія по воронці: установки → авторизація → покупка**

In [28]:
df_app_sessions = pd.read_sql("""SELECT * FROM app_sessions""", engine)
df_product_views_log = pd.read_sql("""SELECT * FROM product_views_log""", engine)
df_devices_users_map = pd.read_sql("""SELECT * FROM devices_users_map""", engine)
df_orders_log = pd.read_sql("""SELECT * FROM orders_log""", engine)

print(df_app_sessions.shape, df_product_views_log.shape, df_devices_users_map.shape, df_orders_log.shape, sep='\n')

(500, 6)
(3233, 4)
(300, 2)
(2559, 3)


In [29]:
installs = df_app_sessions.device_code.nunique()
installs

500

In [30]:
auth_users = df_devices_users_map.user_uuid.count()
auth_users

np.int64(300)

In [31]:
purchases = df_devices_users_map[df_devices_users_map.user_uuid.isin(df_orders_log.user_uuid)].device_code.nunique()
purchases

300

In [32]:
print('установки → авторизація:', f'{auth_users / installs * 100:.2f}%')

установки → авторизація: 60.00%


In [33]:
print('авторизація → покупка:', f'{purchases / auth_users * 100:.2f}%')

авторизація → покупка: 100.00%


8. **Канали, які привели до покупок**

In [22]:
query = """
SELECT app_sessions.acquisition_channel,
       COUNT(DISTINCT orders_log.user_uuid) AS buyers,
       SUM(orders_log.total_uah) AS revenue
FROM app_sessions
JOIN devices_users_map ON app_sessions.device_code = devices_users_map.device_code
JOIN orders_log ON orders_log.user_uuid = devices_users_map.user_uuid
GROUP BY app_sessions.acquisition_channel
ORDER BY revenue DESC;
"""
df = pd.read_sql(query, engine)
df

,acquisition_channel,buyers,revenue
0,Facebook,89,1150615.53
1,Referral,73,994129.97
2,Organic,70,937531.37
3,Google Ads,68,913383.78


9. **Користувачі, що не здійснили жодної покупки**


In [25]:
query = """
SELECT devices_users_map.user_uuid
FROM devices_users_map
LEFT JOIN orders_log ON devices_users_map.user_uuid = orders_log.user_uuid
WHERE orders_log.user_uuid IS NULL;
"""
df = pd.read_sql(query, engine)
df

,user_uuid


10. **Перегляди, але без авторизації**

In [27]:
query = """
SELECT DISTINCT product_views_log.device_code
FROM product_views_log
LEFT JOIN devices_users_map ON devices_users_map.device_code = product_views_log.device_code
WHERE devices_users_map.device_code IS NULL;
"""

df = pd.read_sql(query, engine)
df

,device_code
0,DVC0141
1,DVC0333
2,DVC0209
3,DVC0137
4,DVC0194
...,...
195,DVC0185
196,DVC0182
197,DVC0037
198,DVC0367
